## 1. Notebook completo (celdas)

Celda 1 – Importar librerías y ruta

In [2]:
import pandas as pd
import numpy as np

# Ruta al archivo CSV original
ruta_csv = "transacciones_raw.csv"


Celda 2 – Carga y exploración inicial


In [3]:
# 1. Carga y exploración de datos

df = pd.read_csv(ruta_csv)

print("Primeras filas del dataset original:")
display(df.head())

print("\nInformación del DataFrame:")
df.info()

print("\nEstadísticas descriptivas (numéricas):")
display(df.describe())

print("\nValores nulos por columna:")
print(df.isna().sum())

duplicados = df.duplicated().sum()
print(f"\nCantidad de filas duplicadas: {duplicados}")


Primeras filas del dataset original:


,id_transaccion,id_cliente,fecha,monto,tasa_interes,tipo_transaccion,segmento_cliente,pais
0,1,1001,2024-01-05,2500.50,12.5,pago,retail,Chile
1,2,1002,2024-01-06,150.00,10.0,transferencia,pyme,Chile
2,3,1003,2024-01-06,980.75,15.0,retiro,retail,Peru
3,4,1001,2024-01-07,NaN,12.5,pago,retail,Chile
4,5,1004,2024-01-08,50.00,NaN,deposito,corporativo,Argentina



Información del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_transaccion    25 non-null     int64  
 1   id_cliente        25 non-null     int64  
 2   fecha             25 non-null     object 
 3   monto             23 non-null     float64
 4   tasa_interes      23 non-null     float64
 5   tipo_transaccion  25 non-null     object 
 6   segmento_cliente  24 non-null     object 
 7   pais              25 non-null     object 
dtypes: float64(2), int64(2), object(4)
memory usage: 1.7+ KB

Estadísticas descriptivas (numéricas):


,id_transaccion,id_cliente,monto,tasa_interes
count,25.000000,25.000000,23.000000,23.000000
mean,13.000000,1005.840000,1105.728261,13.043478
std,7.359801,3.954744,1468.442146,3.233043
min,1.000000,1001.000000,40.000000,9.000000
25%,7.000000,1002.000000,135.000000,10.000000
50%,13.000000,1005.000000,450.000000,12.500000
75%,19.000000,1008.000000,1700.000000,15.000000
max,25.000000,1014.000000,6000.000000,20.000000



Valores nulos por columna:
id_transaccion      0
id_cliente          0
fecha               0
monto               2
tasa_interes        2
tipo_transaccion    0
segmento_cliente    1
pais                0
dtype: int64

Cantidad de filas duplicadas: 0


Celda 3 – Limpieza: imputación y eliminación de duplicados

In [4]:
# 2. Limpieza y transformación de datos

df_limpio = df.copy()

# Imputación de columnas numéricas: 'monto' y 'tasa_interes'
for col in ["monto", "tasa_interes"]:
    if col in df_limpio.columns:
        media = df_limpio[col].mean()
        df_limpio[col] = df_limpio[col].fillna(media)
        print(f"Imputada media en columna numérica '{col}': {media}")

# Imputación de columnas categóricas: 'tipo_transaccion' y 'segmento_cliente'
for col in ["tipo_transaccion", "segmento_cliente"]:
    if col in df_limpio.columns:
        moda = df_limpio[col].mode()
        if not moda.empty:
            df_limpio[col] = df_limpio[col].fillna(moda[0])
            print(f"Imputada moda en columna categórica '{col}': {moda[0]}")

# Eliminar registros duplicados
antes = len(df_limpio)
df_limpio = df_limpio.drop_duplicates()
despues = len(df_limpio)
print(f"\nFilas antes de eliminar duplicados: {antes}")
print(f"Filas después de eliminar duplicados: {despues}")
print(f"Duplicados eliminados: {antes - despues}")


Imputada media en columna numérica 'monto': 1105.7282608695652
Imputada media en columna numérica 'tasa_interes': 13.043478260869565
Imputada moda en columna categórica 'tipo_transaccion': transferencia
Imputada moda en columna categórica 'segmento_cliente': retail

Filas antes de eliminar duplicados: 25
Filas después de eliminar duplicados: 25
Duplicados eliminados: 0


Celda 4 – Transformación de categóricas a numéricas

In [5]:
# Conversión de columnas categóricas a variables numéricas

df_modelo = df_limpio.copy()

# Map manual para 'tipo_transaccion'
if "tipo_transaccion" in df_modelo.columns:
    mapa_tipo = {
        "pago": 0,
        "retiro": 1,
        "transferencia": 2,
        "deposito": 3
    }
    df_modelo["tipo_transaccion_cod"] = df_modelo["tipo_transaccion"].map(mapa_tipo)

# One-hot encoding para 'segmento_cliente'
if "segmento_cliente" in df_modelo.columns:
    dummies_segmento = pd.get_dummies(df_modelo["segmento_cliente"], prefix="segmento")
    df_modelo = pd.concat([df_modelo.drop(columns=["segmento_cliente"]), dummies_segmento], axis=1)

print("Columnas del DataFrame transformado:")
print(df_modelo.columns)
display(df_modelo.head())


Columnas del DataFrame transformado:
Index(['id_transaccion', 'id_cliente', 'fecha', 'monto', 'tasa_interes',
       'tipo_transaccion', 'pais', 'tipo_transaccion_cod',
       'segmento_corporativo', 'segmento_pyme', 'segmento_retail'],
      dtype='object')


,id_transaccion,id_cliente,fecha,monto,tasa_interes,tipo_transaccion,pais,tipo_transaccion_cod,segmento_corporativo,segmento_pyme,segmento_retail
0,1,1001,2024-01-05,2500.500000,12.500000,pago,Chile,0,False,False,True
1,2,1002,2024-01-06,150.000000,10.000000,transferencia,Chile,2,False,True,False
2,3,1003,2024-01-06,980.750000,15.000000,retiro,Peru,1,False,False,True
3,4,1001,2024-01-07,1105.728261,12.500000,pago,Chile,0,False,False,True
4,5,1004,2024-01-08,50.000000,13.043478,deposito,Argentina,3,True,False,False


Celda 5 – Optimización y estructuración (groupby, filtros, renombre)

In [6]:
# 3. Optimización y estructuración de datos

df_final = df_modelo.copy()

# Resumen por cliente: total, promedio y cantidad de transacciones
if "id_cliente" in df_final.columns and "monto" in df_final.columns:
    resumen_cliente = (
        df_final
        .groupby("id_cliente")
        .agg(
            monto_total=("monto", "sum"),
            monto_promedio=("monto", "mean"),
            transacciones=("monto", "count")
        )
        .reset_index()
    )
    print("Resumen por cliente:")
    display(resumen_cliente.head())

# Filtro de transacciones de alto monto (ejemplo > 1000)
if "monto" in df_final.columns:
    df_montos_altos = df_final[df_final["monto"] > 1000]
    print("\nTransacciones con monto > 1000:")
    display(df_montos_altos.head())

# Renombrar columnas para mejor interpretación
df_final = df_final.rename(columns={
    "id_cliente": "cliente_id",
    "monto": "monto_transaccion"
})

# Reordenar columnas principales al inicio
columnas_deseadas = [
    col for col in
    ["cliente_id", "id_transaccion", "fecha", "monto_transaccion", "tipo_transaccion", "tasa_interes", "pais"]
    if col in df_final.columns
]
otras_columnas = [c for c in df_final.columns if c not in columnas_deseadas]
df_final = df_final[columnas_deseadas + otras_columnas]

print("\nDataFrame final estructurado:")
display(df_final.head())


Resumen por cliente:


,id_cliente,monto_total,monto_promedio,transacciones
0,1001,3956.228261,1318.742754,3
1,1002,1230.000000,307.500000,4
2,1003,1055.750000,527.875000,2
3,1004,300.000000,150.000000,2
4,1005,3900.000000,1950.000000,2



Transacciones con monto > 1000:


,id_transaccion,id_cliente,fecha,monto,tasa_interes,tipo_transaccion,pais,tipo_transaccion_cod,segmento_corporativo,segmento_pyme,segmento_retail
0,1,1001,2024-01-05,2500.500000,12.5,pago,Chile,0,False,False,True
3,4,1001,2024-01-07,1105.728261,12.5,pago,Chile,0,False,False,True
5,6,1005,2024-01-08,3000.000000,18.0,transferencia,Chile,2,False,True,False
9,10,1008,2024-01-10,2200.000000,16.0,transferencia,Argentina,2,True,False,False
12,13,1009,2024-01-12,1300.000000,11.0,transferencia,Chile,2,False,True,False



DataFrame final estructurado:


,cliente_id,id_transaccion,fecha,monto_transaccion,tipo_transaccion,tasa_interes,pais,tipo_transaccion_cod,segmento_corporativo,segmento_pyme,segmento_retail
0,1001,1,2024-01-05,2500.500000,pago,12.500000,Chile,0,False,False,True
1,1002,2,2024-01-06,150.000000,transferencia,10.000000,Chile,2,False,True,False
2,1003,3,2024-01-06,980.750000,retiro,15.000000,Peru,1,False,False,True
3,1001,4,2024-01-07,1105.728261,pago,12.500000,Chile,0,False,False,True
4,1004,5,2024-01-08,50.000000,deposito,13.043478,Argentina,3,True,False,False


Celda 6 – Ejemplo antes / después para el informe

In [7]:
print("Ejemplo ANTES de la limpieza (dataset original):")
display(df.head())

print("\nEjemplo DESPUÉS de la limpieza y transformación (DataFrame final):")
display(df_final.head())


Ejemplo ANTES de la limpieza (dataset original):


,id_transaccion,id_cliente,fecha,monto,tasa_interes,tipo_transaccion,segmento_cliente,pais
0,1,1001,2024-01-05,2500.50,12.5,pago,retail,Chile
1,2,1002,2024-01-06,150.00,10.0,transferencia,pyme,Chile
2,3,1003,2024-01-06,980.75,15.0,retiro,retail,Peru
3,4,1001,2024-01-07,NaN,12.5,pago,retail,Chile
4,5,1004,2024-01-08,50.00,NaN,deposito,corporativo,Argentina



Ejemplo DESPUÉS de la limpieza y transformación (DataFrame final):


,cliente_id,id_transaccion,fecha,monto_transaccion,tipo_transaccion,tasa_interes,pais,tipo_transaccion_cod,segmento_corporativo,segmento_pyme,segmento_retail
0,1001,1,2024-01-05,2500.500000,pago,12.500000,Chile,0,False,False,True
1,1002,2,2024-01-06,150.000000,transferencia,10.000000,Chile,2,False,True,False
2,1003,3,2024-01-06,980.750000,retiro,15.000000,Peru,1,False,False,True
3,1001,4,2024-01-07,1105.728261,pago,12.500000,Chile,0,False,False,True
4,1004,5,2024-01-08,50.000000,deposito,13.043478,Argentina,3,True,False,False


Celda 7 – Exportación a CSV y Excel

In [8]:
# 4. Exportación de datos

df_final.to_csv("transacciones_limpias.csv", index=False)
df_final.to_excel("transacciones_limpias.xlsx", index=False)

print("Archivos exportados: transacciones_limpias.csv y transacciones_limpias.xlsx")


Archivos exportados: transacciones_limpias.csv y transacciones_limpias.xlsx
